# German Credit Risk — Logistic Regression for Loan Approval

**Authors:** Diana Fernandes, Filipa Neves, Inês Rocha  
**Institution:** Instituto Superior de Engenharia do Porto (ISEP)  
**Course:** Statistical Analysis of Data (ESADA)

---

## Project Overview

When a bank receives a loan application, it must decide whether to approve it based on the applicant's profile. Two types of risk are involved:

- **Business loss:** rejecting a loan for a client who would have repaid it
- **Financial loss:** approving a loan for a client who defaults

The goal of this project is to develop a **logistic regression model** that supports bank managers in making data-driven loan approval decisions, minimising both types of risk.

The **German Credit dataset** contains 1,000 observations and 20 descriptive variables related to the socioeconomic and demographic profiles of loan applicants. The target variable, `Creditability`, classifies each applicant as `Good Credit` or `Bad Credit`.

---


## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from scipy.stats import chi2

warnings.filterwarnings('ignore')

In [ ]:
# Load dataset — update the file path as needed
df = pd.read_csv("german_credit.csv")

print("Dataset shape:", df.shape)
df.info()

The dataset contains **1,000 observations** and **21 columns** (20 predictor variables + 1 target). No missing values were detected.

In [ ]:
# Remove any rows with missing values (none expected in this dataset)
df = df.dropna()

df.head()

In [ ]:
# Descriptive statistics for all variables
df.describe()

## 2. Variable Encoding

Numerical codes are mapped to descriptive labels to improve interpretability during exploratory analysis.


In [ ]:
# Map all categorical variables from numeric codes to descriptive labels

df['Creditability'] = df['Creditability'].astype(object).map({0: "BadCredit", 1: "GoodCredit"})

df['Account Balance'] = df['Account Balance'].astype(object).map({
    1: "<0DM", 2: "<200DM", 3: ">=200DM", 4: "No existing account"
})

df['Payment Status of Previous Credit'] = df['Payment Status of Previous Credit'].astype(object).map({
    0: "no credits taken",
    1: "all credits at this bank paid back duly",
    2: "existing credits paid back duly till now",
    3: "delay in paying off in the past",
    4: "critical account"
})

df['Purpose'] = df['Purpose'].astype(object).map({
    0: "car (new)", 1: "car (used)", 2: "furniture/equipment",
    3: "radio/television", 4: "domestic appliances", 5: "repairs",
    6: "education", 7: "vacation", 8: "retraining", 9: "business", 10: "others"
})

df['Value Savings/Stocks'] = df['Value Savings/Stocks'].astype(object).map({
    1: "<100DM", 2: "<500DM", 3: "<1000DM", 4: ">=1000DM",
    5: "unknown/no savings account"
})

df['Length of current employment'] = df['Length of current employment'].astype(object).map({
    1: "unemployed", 2: "<1year", 3: "<4years", 4: "<7years", 5: ">=7years"
})

df['Sex & Marital Status'] = df['Sex & Marital Status'].astype(object).map({
    1: "male:divorced/separated", 2: "female:divorced/separated/married",
    3: "male:single", 4: "male:married/widowed", 5: "female:single"
})

df['Guarantors'] = df['Guarantors'].astype(object).map({1: "none", 2: "co-applicant", 3: "guarantor"})

df['Most valuable available asset'] = df['Most valuable available asset'].astype(object).map({
    1: "real estate", 2: "building society/life insurance",
    3: "car or other", 4: "unknown/no property"
})

df['Concurrent Credits'] = df['Concurrent Credits'].astype(object).map({1: "bank", 2: "stores", 3: "none"})

df['Type of apartment'] = df['Type of apartment'].astype(object).map({1: "rent", 2: "own", 3: "for free"})

df['Occupation'] = df['Occupation'].astype(object).map({
    1: "unemployed/unskilled-non-resident", 2: "unskilled-resident",
    3: "skilled employee/official",
    4: "management/self-employed/highly qualified"
})

df['Telephone'] = df['Telephone'].astype(object).map({1: "none", 2: "yes"})
df['Foreign Worker'] = df['Foreign Worker'].astype(object).map({1: "yes", 2: "no"})

print("Encoding complete.")
df.info()

In [ ]:
df.head()

## 3. Data Quality Assessment

### 3.1 Categorical Variable Value Counts

Each categorical variable is inspected individually to confirm consistency and absence of unexpected values.


In [ ]:
categorical_cols = [
    'Account Balance', 'Payment Status of Previous Credit', 'Purpose',
    'Value Savings/Stocks', 'Length of current employment',
    'Sex & Marital Status', 'Guarantors', 'Most valuable available asset',
    'Concurrent Credits', 'Type of apartment', 'Occupation', 'Telephone', 'Foreign Worker'
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

### 3.2 Outlier Detection — Numerical Variables

In [ ]:
# Boxplots to visualise distributions and detect outliers in numerical variables
numerical_columns = ['Duration of Credit (month)', 'Credit Amount', 'Age (years)', 'Instalment per cent']

plt.figure(figsize=(14, 10))
for i, column in enumerate(numerical_columns):
    plt.subplot(2, 2, i + 1)
    sns.boxplot(y=df[column], color='lightblue')
    plt.title(column)
    plt.ylabel('Value')

plt.suptitle('Boxplots of Numerical Variables', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 4. Exploratory Data Analysis (EDA)

### 4.1 Distribution of Categorical Variables


In [ ]:
# Bar charts showing the frequency distribution of each categorical variable
categorical_columns = [
    'Creditability', 'Account Balance', 'Payment Status of Previous Credit', 'Purpose',
    'Value Savings/Stocks', 'Length of current employment', 'Sex & Marital Status',
    'Guarantors', 'Most valuable available asset', 'Concurrent Credits',
    'Type of apartment', 'Occupation', 'Telephone', 'Foreign Worker'
]

num_cols = 5
num_rows = (len(categorical_columns) + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(20, num_rows * 6))
axes = axes.flatten()

for i, column in enumerate(categorical_columns):
    df[column].value_counts().plot(kind='bar', ax=axes[i], color='steelblue')
    axes[i].set_title(column, fontsize=9)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Frequency')
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribution of Categorical Variables', fontsize=13)
plt.tight_layout()
plt.show()

### 4.2 Creditability Proportion by Categorical Variable

Stacked bar charts show the proportion of Good Credit and Bad Credit within each category.

**Key findings:**
- `Account Balance`: lower balances are associated with higher Bad Credit rates
- `Foreign Worker`: foreign workers show a higher probability of Bad Credit classification
- `Sex & Marital Status`: no strong variation between categories


In [ ]:
# Stacked bar charts: proportion of Good/Bad Credit within each categorical variable
categorical_columns = [
    "Account Balance", "Payment Status of Previous Credit", "Purpose",
    "Value Savings/Stocks", "Length of current employment", "Sex & Marital Status",
    "Guarantors", "Most valuable available asset", "Concurrent Credits",
    "Type of apartment", "Occupation", "Telephone", "Foreign Worker"
]

num_cols = 4
num_rows = (len(categorical_columns) + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(16, num_rows * 5))
axes = axes.flatten()

for i, column in enumerate(categorical_columns):
    proportions = pd.crosstab(df[column], df['Creditability'], normalize='index')
    proportions.plot(kind='bar', stacked=True, ax=axes[i],
                     color=['salmon', 'lightgreen'], legend=False)
    axes[i].set_title(column, fontsize=9)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Proportion', fontsize=8)
    axes[i].tick_params(axis='x', labelsize=7)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Creditability Proportion by Categorical Variable', fontsize=12)
plt.tight_layout()
plt.figlegend(['Bad Credit', 'Good Credit'], loc='upper center', ncol=2, fontsize=9)
plt.show()

### 4.3 Distribution of Numerical Variables by Creditability

In [ ]:
# Boxplots comparing numerical variable distributions between Good and Bad Credit
numerical_columns = [
    "Duration of Credit (month)", "Credit Amount", "Instalment per cent",
    "Duration in Current address", "Age (years)", "No of Credits at this Bank"
]

num_cols = 3
num_rows = (len(numerical_columns) + num_cols - 1) // num_cols

fig, axes = plt.subplots(num_rows, num_cols, figsize=(15, num_rows * 5))
axes = axes.flatten()

for i, column in enumerate(numerical_columns):
    sns.boxplot(data=df, x='Creditability', y=column,
                palette=['lightgreen', 'salmon'], ax=axes[i])
    axes[i].set_title(column, fontsize=10)
    axes[i].set_xlabel('Creditability', fontsize=8)
    axes[i].set_ylabel(column, fontsize=8)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Numerical Variables by Creditability', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Correlation Analysis

In [ ]:
# Correlation matrix of numerical variables
continuous_vars = df[[
    'Duration of Credit (month)', 'Credit Amount', 'Age (years)',
    'Instalment per cent', 'Duration in Current address',
    'No of Credits at this Bank', 'No of dependents'
]]

corr = continuous_vars.corr()
print(corr)

plt.figure(figsize=(9, 7))
ax = sns.heatmap(corr, vmin=-1, vmax=1, center=0,
                 cmap=sns.diverging_palette(20, 220, n=200), square=True, annot=True, fmt='.2f')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title('Correlation Matrix — All Numerical Variables')
plt.tight_layout()
plt.show()

`Instalment per cent` and `Credit Amount` show a relatively high correlation. Two additional heatmaps are produced to evaluate the impact of removing each variable.

In [ ]:
# Heatmap excluding Credit Amount (to assess impact of correlation)
vars_no_credit = continuous_vars.drop(['Credit Amount'], axis=1)
corr_no_credit = vars_no_credit.corr()

plt.figure(figsize=(8, 6))
ax = sns.heatmap(corr_no_credit, vmin=-1, vmax=1, center=0,
                 cmap=sns.diverging_palette(20, 220, n=200), square=True, annot=True, fmt='.2f')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title('Correlation Matrix — Excluding Credit Amount')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap excluding Instalment per cent (to assess impact of correlation)
vars_no_instalment = df[[
    'Duration of Credit (month)', 'Credit Amount', 'Age (years)',
    'Duration in Current address', 'No of Credits at this Bank', 'No of dependents'
]]
corr_no_instalment = vars_no_instalment.corr()

plt.figure(figsize=(8, 6))
ax = sns.heatmap(corr_no_instalment, vmin=-1, vmax=1, center=0,
                 cmap=sns.diverging_palette(20, 220, n=200), square=True, annot=True, fmt='.2f')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
plt.title('Correlation Matrix — Excluding Instalment per cent')
plt.tight_layout()
plt.show()

## 6. Logistic Regression Modelling

### 6.1 Feature Engineering

One-hot encoding is applied to all categorical predictors. Numerical variables are kept as-is.
The target variable `Creditability` is mapped to binary (0 = Bad Credit, 1 = Good Credit).


In [ ]:
# Separate features from target variable
df1 = df.drop("Creditability", axis=1)

# Split into categorical and numerical subsets
categorical_vars = df1.select_dtypes('object')
numerical_vars = df1.select_dtypes('int64')

# Apply one-hot encoding (drop_first=True to avoid multicollinearity)
categorical_dummies = pd.get_dummies(categorical_vars, drop_first=True).astype(int)

# Combine numerical and encoded categorical variables
df_combined = pd.concat([numerical_vars, categorical_dummies], axis=1)

# Define features (X) and binary target (y)
X = df_combined
Y = df["Creditability"].map({"BadCredit": 0, "GoodCredit": 1})

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", Y.value_counts())

In [ ]:
# Stratified train/test split (80/20) to preserve class balance
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.20, random_state=40, stratify=Y
)

X_train = X_train.astype(int)
Y_train = Y_train.astype(int)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

### 6.2 Model 1 — Full Logistic Regression

All available features are included. This model serves as the baseline and identifies statistically significant predictors.


In [ ]:
# Fit logistic regression using all features (statsmodels for p-values and summary)
sm_model_1 = sm.Logit(Y_train, sm.add_constant(X_train)).fit(disp=0)

print("P-values for all predictors:")
print(sm_model_1.pvalues.sort_values())

sm_model_1.summary()

**Model 1 Results:**
- Global p-value: 1.387e-30 — model is statistically significant
- Pseudo R²: 0.2643 — ~26% of variance in the target explained
- Log-Likelihood: -359.55

**Significant variables** (p-value < 0.05): Duration of Credit, Credit Amount, Instalment per cent, Account Balance (>=200DM, No existing account), Payment Status (critical account, delay in past), Purpose (car used), Employment length (<7 years), Sex & Marital Status (male:single), Guarantors (guarantor), Foreign Worker (yes).


### 6.3 Model 2 — Reduced Model (Significant Variables Only)

Only the variables identified as significant in Model 1 are retained. This reduces complexity and helps prevent overfitting.


In [ ]:
# Select only statistically significant variables from Model 1
significant_features = [
    'Duration of Credit (month)', 'Credit Amount', 'Account Balance_>=200DM',
    'Account Balance_No existing account', 'Instalment per cent',
    'Payment Status of Previous Credit_critical account',
    'Payment Status of Previous Credit_delay in paying off in the past',
    'Purpose_car (used)', 'Length of current employment_<7years',
    'Sex & Marital Status_male:single', 'Guarantors_guarantor', 'Foreign Worker_yes'
]

# Filter to only include columns that exist in training data
significant_features = [f for f in significant_features if f in X_train.columns]

X_train_m2 = X_train[significant_features]
sm_model_2 = sm.Logit(Y_train, sm.add_constant(X_train_m2)).fit(disp=0)

print(sm_model_2.summary())

**Model 2 Results:**
- Global p-value: 1.367e-32
- Pseudo R²: 0.1866 — slightly lower than Model 1
- Log-Likelihood: -397.49

**Benefits:** simpler model, reduced risk of overfitting, easier interpretation. Trade-off: slightly lower explanatory power.


### 6.4 Model 3 — Final Model (Removing Correlated Variable)

`Instalment per cent` is removed from Model 2 due to its moderate correlation with `Credit Amount`, reducing multicollinearity.


In [ ]:
# Remove Instalment per cent to address multicollinearity with Credit Amount
significant_features_m3 = [
    'Duration of Credit (month)', 'Credit Amount', 'Account Balance_>=200DM',
    'Account Balance_No existing account',
    'Payment Status of Previous Credit_critical account',
    'Payment Status of Previous Credit_delay in paying off in the past',
    'Purpose_car (used)', 'Length of current employment_<7years',
    'Sex & Marital Status_male:single', 'Guarantors_guarantor', 'Foreign Worker_yes'
]

significant_features_m3 = [f for f in significant_features_m3 if f in X_train.columns]

X_train_m3 = X_train[significant_features_m3]
sm_model_3 = sm.Logit(Y_train, sm.add_constant(X_train_m3)).fit(disp=0)

print(sm_model_3.summary())

**Model 3 Results:**
- Global p-value: 2.206e-32
- Pseudo R²: 0.1825
- Log-Likelihood: -399.50

**Model Comparison:**

| Model | Variables | Pseudo R² | Log-Likelihood | Multicollinearity |
|---|---|---|---|---|
| Model 1 | All features | 0.2643 | -359.55 | Present |
| Model 2 | Significant only | 0.1866 | -397.49 | Reduced |
| Model 3 | Without Instalment % | 0.1825 | -399.50 | None |

**Selected model: Model 3** — simplest, most interpretable, free of multicollinearity.


## 7. Odds Ratios Interpretation

In [ ]:
# Calculate and display odds ratios with 95% confidence intervals
odds_ratios = pd.DataFrame({
    "OR": sm_model_3.params,
    "Lower CI": sm_model_3.conf_int()[0],
    "Upper CI": sm_model_3.conf_int()[1],
})
odds_ratios = np.exp(odds_ratios)
print(odds_ratios)

**Key interpretations:**
- **Duration of Credit (month):** OR < 1 — longer loan duration decreases the probability of good creditability
- **Account Balance >= 200DM:** OR ~4.27 — applicants with this balance are over 4× more likely to be classified as Good Credit
- **Foreign Worker (yes):** OR < 1 — foreign workers have approximately 22% lower odds of Good Credit
- Variables whose confidence interval contains 1 are not statistically significant


## 8. Model Evaluation

In [ ]:
# Prepare test set with the same features used in Model 3
X_test_m3 = X_test[[f for f in significant_features_m3 if f in X_test.columns]]
X_test_m3_const = sm.add_constant(X_test_m3)

# Generate predictions
p_estimate = sm_model_3.predict(X_test_m3_const)
prediction = np.round(p_estimate)

# Confusion matrix
cm = confusion_matrix(Y_test, prediction)
print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(colorbar=False)
plt.title('Confusion Matrix — Model 3')
plt.show()

In [ ]:
# Performance metrics
TP = cm[1, 1]
TN = cm[0, 0]
FP = cm[0, 1]
FN = cm[1, 0]

accuracy    = (TP + TN) / float(TP + FN + TN + FP)
error       = (FP + FN) / float(TP + FN + TN + FP)
sensitivity = TP / float(TP + FN)
specificity = TN / float(TN + FP)
precision   = TP / float(TP + FP)
recall      = TP / float(TP + FN)
f1          = 2 * TP / float(2 * TP + FP + FN)

print(f"Accuracy:    {accuracy:.4f}")
print(f"Error:       {error:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1 Score:    {f1:.4f}")

**Results interpretation:**
- **Accuracy (72.5%):** correct overall prediction rate
- **Sensitivity (88.57%):** the model correctly identifies 88.6% of Good Credit applicants
- **Specificity (35%):** the model struggles to identify Bad Credit applicants — a known limitation of imbalanced datasets
- **F1 Score (81.55%):** good balance between precision and sensitivity


In [ ]:
# AUC-ROC score
roc_auc = roc_auc_score(Y_test, prediction)
print(f'AUC-ROC: {roc_auc:.3f}')

**AUC-ROC = 0.618** — the model correctly ranks a random positive observation above a negative one 61.8% of the time.

## 9. Model Assumptions and Goodness of Fit

**Logistic regression assumptions verified:**
1. Binary response variable — satisfied (`Creditability`: 0/1)
2. Independent observations — assumed given dataset structure
3. No severe multicollinearity — addressed by removing `Instalment per cent`
4. No extreme outliers — confirmed in EDA
5. Linear relationship between predictors and log-odds — assessed via Hosmer-Lemeshow test
6. Sufficient sample size — 1,000 observations with at least 10 cases per class per predictor

### 9.1 Hosmer-Lemeshow Goodness of Fit Test

- H₀: Observed and predicted event rates are similar across 10 intervals
- H₁: They differ significantly
- If p-value > 0.05, we fail to reject H₀ — model fits the data well


In [ ]:
# Hosmer-Lemeshow goodness of fit test (k=10 deciles)

X_train_m3_const = sm.add_constant(X_train_m3)
p_estimate_train = sm_model_3.predict(X_train_m3_const)

hl_df = pd.DataFrame({"P_i": p_estimate_train, "Outcome": Y_train})
hl_df["decile"] = pd.qcut(hl_df["P_i"], 10)

# Observed and expected events per decile
obs_1 = hl_df["Outcome"].groupby(hl_df["decile"]).sum()
obs_0 = hl_df["Outcome"].groupby(hl_df["decile"]).count() - obs_1
exp_1 = hl_df["P_i"].groupby(hl_df["decile"]).sum()
exp_0 = hl_df["P_i"].groupby(hl_df["decile"]).count() - exp_1

# Hosmer-Lemeshow statistic
hl_stat = (((obs_0 - exp_0) ** 2) / exp_0).sum() + (((obs_1 - exp_1) ** 2) / exp_1).sum()
p_value = 1 - chi2.cdf(hl_stat, df=8)  # degrees of freedom = k - 2 = 8

print(f"Hosmer-Lemeshow statistic: {hl_stat:.4f}")
print(f"P-value: {p_value:.4f}")

if p_value > 0.05:
    print("Result: p-value > 0.05 — the model fits the data well (fail to reject H0).")
else:
    print("Result: p-value <= 0.05 — model fit is questionable (reject H0).")

## 10. Conclusions

This project developed three logistic regression models to predict credit risk for bank loan applicants using the German Credit dataset.

**Model 3** was selected as the final model due to its balance of simplicity, interpretability and statistical robustness:
- Removes multicollinearity by excluding `Instalment per cent`
- Retains 11 statistically significant predictors
- Achieves 72.5% accuracy and 81.55% F1 Score on the test set
- Passes the Hosmer-Lemeshow goodness of fit test (p-value > 0.05)

**Key predictors of Good Credit:**
- High account balance (>= 200 DM) — strongest positive predictor
- Good prior payment status
- Used car purchase purpose
- Longer current employment

**Key predictors of Bad Credit:**
- Longer loan duration
- Foreign worker status
- History of delayed payments or critical account status

The model can serve as a decision-support tool for bank managers, reducing financial risk while maintaining a strong sensitivity to genuine creditworthy applicants.
